In [ ]:
# BBO Capstone – Gaussian Process with UCB

# This notebook demonstrates a simple Bayesian Optimisation loop using
# Gaussian Process regression and an Upper Confidence Bound acquisition
# function. Weekly inputs and outputs are parsed, appended to the dataset,
# and used to propose the next query point.

In [1]:
import re
import os

BASE = "../Data"   # DAta

text = """
This week's input values:
Function 1:	[0.915602, 0.149350]
Function 2:	[0.455221, 0.153277]
Function 3:	[0.600721, 0.978625, 0.810242]
Function 4:	[0.397887, 0.450882, 0.383693, 0.413811]
Function 5:	[0.968901, 0.797949, 0.929774, 0.996142]
Function 6:	[0.410048, 0.441613, 0.835606, 0.859350, 0.003646]
Function 7:	[0.017394, 0.234162, 0.353566, 0.265762, 0.475614, 0.604470]
Function 8:	[0.057036, 0.191383, 0.152787, 0.011185, 0.689367, 0.771824, 0.077298, 0.995820]

This week's output values:

Function 1:	1.3485554759723437e-217
Function 2:	0.039577056064363736
Function 3:	-0.09330413426689704
Function 4:	-0.1931636502363534
Function 5:	4550.616174569862
Function 6:	-0.46015533164724776
Function 7:	1.7668330601134028
Function 8:	9.8479812642185


"""


def parse_all_functions(txt):
    # finding all input blocks
    input_blocks = re.findall(
        r"Function\s+(\d+):\s*\[([0-9\.\,\s\-eE]+)\]",
        txt
    )

    #  finding all output blocks
    output_blocks = re.findall(
        r"Function\s+(\d+):\s*([0-9\.\-eE]+)",
        txt
    )

    # Strings to  numeric 
    parsed_inputs = {}
    for func_id, vec in input_blocks:
        arr = np.fromstring(vec, sep=",")
        parsed_inputs[int(func_id)] = arr

    parsed_outputs = {}
    for func_id, val in output_blocks:
        parsed_outputs[int(func_id)] = float(val)

    
    results = {}
    for i in range(1, 9):
        if i in parsed_inputs and i in parsed_outputs:
            results[i] = (parsed_inputs[i], parsed_outputs[i])

    return results



def append_to_dataset(base_dir, func_id, x_new, y_new):
    folder = os.path.join(base_dir, f"function{func_id}")
    X_path = os.path.join(folder, "initial_inputs.npy")
    Y_path = os.path.join(folder, "initial_outputs.npy")

    X = np.load(X_path)
    Y = np.load(Y_path)

    X_new = np.vstack([X, x_new.reshape(1, -1)])
    Y_new = np.hstack([Y, np.array([y_new])])

    np.save(X_path, X_new)
    np.save(Y_path, Y_new)

    print(f"✅ Function {func_id} appended → X:{X_new.shape}, Y:{Y_new.shape}")


parsed = parse_all_functions(text)

for func_id, (xin, yout) in parsed.items():
    append_to_dataset(BASE, func_id, xin, yout)

print("\n✅ DONE)


✅ Function 1 eklendi → X:(21, 2), Y:(21,)
✅ Function 2 eklendi → X:(21, 2), Y:(21,)
✅ Function 3 eklendi → X:(26, 3), Y:(26,)
✅ Function 4 eklendi → X:(41, 4), Y:(41,)
✅ Function 5 eklendi → X:(31, 4), Y:(31,)
✅ Function 6 eklendi → X:(31, 5), Y:(31,)
✅ Function 7 eklendi → X:(41, 6), Y:(41,)
✅ Function 8 eklendi → X:(51, 8), Y:(51,)

✅ TÜM HAFTALIK VERİLER BAŞARIYLA EKLENDİ!


In [ ]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from scipy.stats import qmc

X = np.load("/function8/initial_inputs.npy")
y = np.load("/function8/initial_outputs.npy")

print("X shape:", X.shape)
print("Y shape:", y.shape)
print("Last X row:", X[-1])
print("Last Y value:", y[-1])

In [ ]:
def predict_next_point(X, y, n_candidates=4096):
    d = X.shape[1] 

    # GP
    kernel = ConstantKernel(1.0) * Matern(nu=2.5) + WhiteKernel(1e-4)

    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True)
    gp.fit(X, y)

    # Sobol candidatre
    sampler = qmc.Sobol(d, scramble=True)
    cand = sampler.random(n_candidates)

    # GP prediction
    mu, sigma = gp.predict(cand, return_std=True)

    # UCB
    beta = 2.0
    ucb = mu + beta * sigma

    best_idx = np.argmax(ucb)
    best_point = cand[best_idx]

    return best_point, ucb[best_idx]

x22, score = predict_next_point(X, y)

print("22.point =", x22)
print("UCB score =", score)

In [ ]:
## Conclusion

#This notebook focuses on a transparent and stable optimisation strategy
#rather than aggressive score chasing. The relatively stable outputs
#reflect convergence toward consistent regions under a limited query budget.
